# Tutorial: Estimating Heterogeneous Contact Matrices with the High-resolution BRCfine Model

Welcome to this tutorial on estimating age-specific social contact matrices using the High-resolution BRC (HiBRCfine) model.

This tutorial walks through the complete workflow of estimating high-resolution, subgroup-stratified social contact matrices. We cover synthetic data generation, model specification, Bayesian inference, and performance evaluation.

## Table of Contents

1. **[Background](#i-background)** - Understanding social contact matrices and the BRC model framework
2. **[Generating Synthetic Contact Data](#ii-generating-synthetic-contact-data)** - Creating realistic contact survey data for validation
	- Loading population and template data
	- Defining study populations with different contact patterns
	- Generating participant and contact datasets
3. **[Fitting the BRC Model](#iii-fitting-the-brc-model)** - Bayesian inference with NumPyro
	- Data configuration and validation
	- Prior specification (Bayesian P-Splines)
	- Stochastic Variational Inference (SVI)
4. **[Analyzing the Posterior](#iv-analyzing-the-posterior)** - Extracting and visualizing results
	- Posterior summaries for contact matrices
	- Marginal contact intensity patterns
5. **[Evaluating Inference Accuracy](#v-evaluate-the-accuracy-of-the-inference)** - Model validation against ground truth


## Setup

In [ ]:
# Core scientific computing
import numpy as np
import pandas as pd
import jax.numpy as jnp
from jax import random

# cntmosaic utilities and data
from cntmosaic.utils import save_tutorial_figure
from cntmosaic.datasets import load_age_distribution, load_template_patterns

# Data generation
from cntmosaic.sim import (
    ParticipantGenerator,
    MatrixGenerator,
    ContactGenerator,
    Subgroup,
)

# Data preprocessing
from cntmosaic.dataloader import CoordToColumns, DataLoader, PopulationProportion

# Modeling
from cntmosaic.models import HiBRCfine
from cntmosaic.models.priors import PSpline2D

# Analysis and visualization
from cntmosaic.analysis import ModelSummariserBRC, ModelEvaluatorBRC
from cntmosaic.vis import plot_mosaic, plot_mosaic_marginal

# Visualization
import altair as alt

# Set random seed for reproducibility
np.random.seed(42)

## I. Background

Social contact data provides essential information for understanding how airborne infectious diseases propagate through populations. A primary statistical object of interest is the **social contact matrix**, which encodes the average frequency or intensity of contacts between individuals of different age groups. The **Bayesian Rate Consistency (BRC)** model is a Bayesian hierarchical model specifically designed to infer high-resolution (i.e., single-year age resolution) contact matrices from contact survey data.

### Model Overview

Suppose we wish to quantify the average number of contacts between individuals of age $a$ and age $b$, where $a,b \in \mathcal{A} = \{0,\ldots,A_{\max}\}$. Further assume the population comprises $J$ disjoint subpopulations indexed by $k \in \{1,\ldots,J\}$.

Let $Y^{k}_{a,b}$ denote the random variable for the total number of reported contacts between individuals of age $a$ in subpopulation $k$ with individuals of age $b$, given $N^k_a$ survey participants of age $a$ in subpopulation $k$. The **HiBRCfine** model postulates the following statistical model for $Y^{k}_{a,b}$:

$$
\begin{align*}
Y^{k}_{a,b} &\sim \operatorname{NegBin}(\mu^{k}_{a,b}, \varphi) \\
\log \mu^{k}_{a,b} &= \beta_0 + f(a,b) + \log \delta^{k}_{a,b} + \log N^k_a + \log P_b
\end{align*}
$$

**Model Components:**

- $\beta_0$: Global intercept parameter with prior $\mathcal{N}(-\log \overline{P}, 2.5^2)$, where $\overline{P} = \frac{1}{A_{\max}+1}\sum_b P_b$
- $f(a,b)$: Smooth 2D function representing the log contact rate structure (specified via a spatial prior)
- $\log \delta^{k}_{a,b}$: Log-scale offset term modeling subpopulation-specific deviations from the global pattern
- $\log N^k_a$: Log sample size offset (number of participants of age $a$ in subpopulation $k$)
- $\log P_b$: Log population size of age $b$ (converts rates to intensities)
- $\varphi$: Negative binomial overdispersion parameter with prior $\varphi \sim \text{Gamma}(2, 0.1)$

### Interpretation of Quantities

The model enables inference on three related quantities:

**1. Contact Rate** (per-capita contacts):
$$
\log \gamma_{a,b} := \beta_0 + f(a,b)
$$
This represents the population-averaged log rate of contacts between age $a$ and age $b$, independent of population structure.

**2. Contact Intensity** (expected number of contacts):
$$
\log m_{a,b} := \beta_0 + f(a,b) + \log P_b = \log \gamma_{a,b} + \log P_b
$$
This is the **primary quantity of interest**: the average number of contacts an individual of age $a$ has with individuals of age $b$. This incorporates population age structure through $P_b$.

**3. Subpopulation-Specific Contact Intensity**:
$$
\log m^k_{a,b} := \log m_{a,b} + \log \delta^{k}_{a,b}
$$
The offset $\log \delta^{k}_{a,b}$ captures how subpopulation $k$'s contact patterns deviate from the population average. The model enforces compositional constraints on these offsets (they must sum to zero in an appropriate sense) through the prior specification.


## II. Generating Synthetic Contact Data

While real contact survey data (like POLYMOD) is ideal for practical applications, we use `cntmosaic`'s data generation tools to create realistic synthetic data for this tutorial. Working with synthetic data offers several advantages:

- **Ground truth validation**: Known true contact patterns allow us to evaluate model accuracy
- **Controlled experiments**: Systematic variation of sample sizes, contact intensities, and subpopulation structures
- **Prior sensitivity analysis**: Assessment of how different prior choices affect inference quality
- **Reproducibility**: Consistent results across different users and platforms

### Synthetic Data Workflow

The data generation process mirrors the structure of real contact surveys:

1. **Define population structure**: Specify age distribution and subpopulation characteristics
2. **Generate participants**: Sample survey participants according to the defined population
3. **Generate contact matrices**: Create realistic age-age contact patterns using template mixing
4. **Simulate survey responses**: Sample contacts for each participant based on true contact matrices

**Template-based approach**: Rather than specifying contact patterns manually, we use empirically-derived "template matrices" from Mistry et al. (2021). These templates represent contact patterns in four distinct social settings—household, workplace, school, and community—based on real demographic data. By randomly mixing these templates with varying weights, we generate diverse yet realistic contact patterns.

The following subsections walk through each step in detail.


### 2.1 Load Population and Template Data

We require two foundational inputs before generating synthetic contact data:

1. **Template contact matrices**: Empirically-derived age-structured contact patterns from Mistry et al. (2021) representing four distinct social contexts:
   - **Household**: Family and co-residential contacts
   - **Workplace**: Professional and workplace interactions  
   - **School**: Educational setting contacts (daycare, K-12, university)
   - **Community**: Other settings (shopping, leisure, transportation)

2. **Population age distribution**: The demographic structure of the population

The `cntmosaic.datasets` module provides convenient loaders for both:


In [ ]:
# Load contact pattern templates
templates = load_template_patterns(country="United_States", max_age=80)

# Load reference population age distribution
df_age_dist = load_age_distribution(country="United_States", max_age=80)

# Extract the population size counts for each age
age_dist = df_age_dist.P.values

**Visualizing template data**: Before generating synthetic contact patterns, let's examine the empirical templates and population structure. The four template matrices below show distinct age-mixing patterns characteristic of each social setting:

- **Household**: Strong diagonal patterns (within-family contacts) plus off-diagonal clusters (parent-child contacts)
- **Workplace**: Broad rectangular patterns concentrated in working ages (18-65 years)
- **School**: Sharp diagonal bands corresponding to age-grade cohorts (5-18 years)
- **Community**: Diffuse patterns reflecting diverse public interactions across all ages

These templates will be randomly weighted and combined to create realistic synthetic contact matrices for each subgroup.


In [ ]:
# Plot template matrices
charts = []
for key, template in templates.items():
    chart = plot_mosaic(
        template, title=f"Template matrix for: {key.capitalize()}"
    ).properties(width=200, height=200)
    charts.append(chart)

alt.hconcat(*charts).resolve_scale(color="independent")

### 2.2 Define Study Population

We define the subpopulations in our synthetic study using the `Subgroup` dataclass from `cntmosaic.sim`. This dataclass specifies both **population characteristics** (age distribution, contact behavior) and **survey sampling design** (number of participants to recruit).

**Understanding `Subgroup` parameters**:

```python
Subgroup(
    n=500,                              # Survey participants to recruit from this subgroup
    age_dist=np.array([500, 200, 300]), # Population age distribution (actual counts)
    mean_cint_margin=10.0,              # Average daily contacts per person
    label="example-subgroup"            # Identifier for this subgroup
)
```

This example defines a subgroup with:
- **Population structure**: Three age groups with 500, 200, and 300 individuals respectively
- **Survey sampling**: 500 participants will be recruited from this population (proportional to `age_dist`)
- **Contact behavior**: Average of 10 contacts per person per day

**Our tutorial setup**: We define three subgroups with different contact intensities (low, medium, high) to demonstrate how the model recovers heterogeneous contact patterns:


In [ ]:
N_PART = 1500  # Total number of participants in the study
MCINT_MEANS = [
    5.0,
    10.0,
    15.0,
]  # Average contacts per person per day in each sub-population
LABELS = ["low", "mid", "high"]

subgroups = [
    Subgroup(
        n=N_PART // len(MCINT_MEANS),  # Placeholder, will be set later
        age_dist=age_dist // len(MCINT_MEANS),  # Follow US age distribution
        mean_cint_margin=mean,  # Mean contacts per person per day
        label=label,  # Population label
    )
    for mean, label in zip(MCINT_MEANS, LABELS)
]

print(
    f"Defined {len(MCINT_MEANS)} subgroups with mean marginal contact intensities: {MCINT_MEANS}"
)

### 2.3 Generate Participant Data

We now sample survey participants using the `ParticipantGenerator` class. This simulates the participant recruitment phase of a contact survey, generating a roster of individuals who will report their contacts.

**How it works**:
1. Takes a list of `Subgroup` objects as input
2. Samples `n` participants from each subgroup proportional to `age_dist`
3. Returns a DataFrame with participant characteristics: `id`, `age_group`, and `subgroup`


In [ ]:
# Initialize participant generator
part_gen = ParticipantGenerator(subgroups)

# Generate participants
df_part = part_gen.generate(seed=42)
df_part["subgroup"] = pd.Categorical(
    df_part["subgroup"], categories=LABELS, ordered=True
)

print(f"Number of participants: {len(df_part)}")
print(f"\nFirst 10 participants:")

# Show the first 10
df_part.head(10)

### 2.4 Generate True Contact Matrices

We now generate realistic contact intensity matrices for each subgroup by randomly mixing the template patterns from section 2.1. The `MatrixGenerator` class provides three generation methods:

- **`generate_single()`**: Creates one matrix (no subgroup stratification)
- **`generate_partial()`**: Creates subgroup-specific matrices stratified by participant's subgroup only
- **`generate_full()`**: Creates matrices stratified by both participant and contact subgroups

**Partial vs. Full stratification**:

The distinction reflects real-world data collection constraints:

- **Partial case** (our tutorial): Survey participants report their subgroup (e.g., "low contact lifestyle"), but the subgroup of each contacted individual is unknown. This yields contact matrices $m^k_{a,b}$ stratified by participant subgroup $k$ only.

- **Full case**: Both participant and contact subgroups are recorded, yielding matrices $m^{k,\ell}_{a,b}$ for all subgroup pairs $(k, \ell)$.

**Template mixing**: `MatrixGenerator` randomly weights the four templates (household, work, school, community) for each subgroup, then scales to match the target `mean_cint_margin` specified in each `Subgroup`.


In [ ]:
# Initialize matrix generator with templates
matrix_gen = MatrixGenerator(templates)

# Generate a realistic contact intensity matrix
contact_matrices = matrix_gen.generate_partial(subgroups, seed=42)  # Dictionary

for label, matrix in contact_matrices.items():
    print(
        f"Subgroup '{label}' mean contacts per person: {matrix.sum(axis=-1).mean():.2f}"
    )

In [ ]:
# Visualize the generated matrices
charts = []
for label, matrix in contact_matrices.items():
    chart = plot_mosaic(
        matrix,
        title=f"True Contact Intensity: {label.capitalize()}",
        zlabel="intensity",
    ).properties(width=200, height=200)
    charts.append(chart)

chart_true = alt.hconcat(*charts).resolve_scale(color="independent")
chart_true

### 2.5 Generate Contact Data

We now simulate the survey response phase where each participant reports their contacts. The `ContactGenerator` class samples contact counts based on the true contact matrices from section 2.4, adding realistic individual-level heterogeneity.

**Data generating process**: For each participant $i$ of age $a_i$ in subgroup $k_i$, the number of contacts reported with individuals of age $b$ is generated as:

$$
Y_{i,b} \sim \text{Poisson}(m^{k_i}_{a_i,b} \cdot \gamma_i), \quad \gamma_i \sim \text{Gamma}(5, 5)
$$

**Model components**:
- $m^{k_i}_{a_i,b}$: True contact intensity (expected contacts for age $a_i$ with age $b$ in subgroup $k_i$)
- $\gamma_i$: Individual random effect capturing person-to-person variability
- Poisson likelihood: Count model for discrete contact events

**Output format**: The `generate()` method returns a long-form DataFrame with one row per (participant, contact age) pair:
- `id`: Participant identifier (links to `df_part`)
- `age_cnt`: Age of contacted individuals
- `y`: Number of contacts reported at that age

This mimics real survey data where participants enumerate contacts by age, which will be aggregated into the $Y^k_{a,b}$ counts used in the BRC model.


In [ ]:
cnt_gen = ContactGenerator(
    df_part=df_part,
    cint_matrices=contact_matrices,
    model="poisson",  # Contact counts follow Poisson distribution
    random_effects=True,  # Whether to include random effects
)

# Generate contact data
df_cnt = cnt_gen.generate(seed=42)

print(f"Total contacts generated: {len(df_cnt)}")
print(f"Average contacts per participant: {len(df_cnt) / len(df_part):.2f}")
df_cnt.head(10)

We can visualize the generated contact data using the `get_contact_matrix_empirical()` method available through the `ContactGenerator` class.


In [ ]:
# Obtain empirical matrices
emp_cint_matrices = cnt_gen.get_contact_matrix_empirical(df_cnt, normalize=True)

charts = []
# Sort by the desired order: low, mid, high
for label in ["low", "mid", "high"]:
    emp_cint_matrix = emp_cint_matrices[label]
    chart = plot_mosaic(
        emp_cint_matrix,
        title=f"Empirical Contact Intensity: {label.capitalize()} subgroup",
        zlabel="Intensity",
    ).properties(width=200, height=200)
    charts.append(chart)

chart_emp = alt.hconcat(*charts).resolve_scale(color="independent")
chart_emp

## III. Fitting the HiBRCfine Model


With our synthetic contact data generated, we demonstrate the complete workflow for fitting the **HiBRCfine** model. This section covers:

1. **Data configuration** (3.1): Mapping DataFrame columns to model inputs
2. **Model specification** (3.2): Choosing priors for contact rates and subgroup offsets  
3. **Inference** (3.3): Running Stochastic Variational Inference (SVI) to obtain approximate posteriors

**What makes HiBRCfine different?** Unlike the base `BRCfine` model (which assumes a single homogeneous population), `HiBRCfine` explicitly models **heterogeneous subgroups** through the offset terms $\log \delta^k_{a,b}$. This allows estimation of how contact patterns differ across subpopulations (e.g., high vs. low contact lifestyles) while sharing information about the global contact structure.

**Computational strategy**: Given the high dimensionality of age-structured contact data (81×81 age pairs × 3 subgroups = 19,683 parameters for offsets alone), we use **SVI** as our primary inference method. This provides scalable approximate inference suitable for both exploration and production use. MCMC remains an alternative for exact uncertainty quantification when computational resources permit.


### 3.1 Configuring the Data

We must map the columns in our DataFrames (`df_part`, `df_cnt`, `df_age_dist`) to the model's expected inputs. The `cntmosaic` package provides two configuration classes for this purpose:

#### `CoordToColumns`: Basic column mapping

This dataclass tells the `DataLoader` which columns contain participant ages, contact ages, IDs, and stratification variables:


In [ ]:
coord_to_cols = CoordToColumns(
    age_part="age_group",  # Column with participant ages
    age_cnt="age_cnt",  # Column with contact ages (or age bins)
    grp_vars_part="subgroup",  # Stratification variable (our 'low'/'mid'/'high' labels)
    id_var="id",  # Unique participant identifier
    age_pop="age",  # Age column in population DataFrame
    size_pop="P",  # Population size/proportion column
)

#### `PopulationProportion`: Subgroup population structure

To estimate subgroup-specific contact patterns, the model requires the population age distribution *for each subgroup*. The `PopulationProportion` class encapsulates this information.

**Simplifying assumption**: For this tutorial, we assume each subgroup comprises exactly 1/3 of the population. In real applications, you would use empirical estimates from census or survey data.


In [ ]:
# Create population DataFrames for each subgroup
df_pop_low = df_age_dist.copy()
df_pop_low["P"] = df_pop_low["P"] / 3  # Assume equal subgroup sizes (1/3 each)
df_pop_low["subgroup"] = "low"

df_pop_mid = df_age_dist.copy()
df_pop_mid["P"] = df_pop_mid["P"] / 3
df_pop_mid["subgroup"] = "mid"

df_pop_high = df_age_dist.copy()
df_pop_high["P"] = df_pop_high["P"] / 3
df_pop_high["subgroup"] = "high"

# Concatenate and ensure categorical ordering
df_subgrp_pop = pd.concat([df_pop_low, df_pop_mid, df_pop_high], ignore_index=True)
df_subgrp_pop["subgroup"] = pd.Categorical(
    df_subgrp_pop["subgroup"], categories=LABELS, ordered=True
)

# Create mapping
pp = PopulationProportion.from_counts(
    df_subgrp_pop, age_col="age", count_col="P", stratify_by="subgroup"
)

#### `DataLoader`: Validation and preprocessing

We now initialize the `DataLoader`, which:

1. **Validates** that all required columns exist and contain appropriate data types
2. **Handles edge cases** like missing ages, zero counts, or unrepresented age groups
3. **Creates model inputs** by converting DataFrames to xarray Datasets and ultimately JAX arrays


In [ ]:
dataloader = DataLoader(
    col_map=coord_to_cols, part=df_part, cnt=df_cnt, pop=df_age_dist, pop_prop=pp
)

### 3.2 Specifying the HiBRCfine Model

We now define the Bayesian priors for the two key components of the model:

1. **Global contact rate** $f(a,b)$: The population-averaged log contact rate structure
2. **Subgroup offsets** $\log \delta^{k}_{a,b}$: Deviations from the global rate for each subgroup

#### Available Prior Families

The `cntmosaic.models.priors` module provides several 2D spatial priors:

- `PSpline2D` (Bayesian P-Splines): Tensor-product B-splines with difference penalties—our **recommended choice** for strong smoothness
- `IGMRF2D` (Intrinsic Gaussian Markov Random Field): Used in van de Kassteele et al. (2017)—good for local smoothing
- `HSGP2D` (Hilbert Space Gaussian Process): Approximate GP from Riutort-Mayol et al. (2023)—assumes stationarity
- `vdKassteele`: The complete hierarchical model from van de Kassteele et al. (2017)

#### Brief Mathematical Overview

**Basis representation**: We model $f(a,b)$ using tensor-product B-splines:

$$
f(a, b) = \sum_{j=1}^{M_1} \sum_{k=1}^{M_2} \beta_{j,k} \cdot \phi_{1j}(a) \cdot \phi_{2k}(b)
$$

where $\phi_{1j}(\cdot)$ and $\phi_{2k}(\cdot)$ are B-spline basis functions. In matrix form:

$$
\mathbf{f} = \Phi \boldsymbol{\beta}, \quad \Phi = \Phi_1 \otimes \Phi_2
$$

where $\Phi_1 \in \mathbb{R}^{A \times M_1}$ and $\Phi_2 \in \mathbb{R}^{A \times M_2}$ are 1D basis matrices, and $\otimes$ denotes the Kronecker product.

**Penalty structure**: To prevent overfitting, we penalize differences between adjacent coefficients using a 2D precision matrix:

$$
Q = \tau (I_{M_2} \otimes D_1^T D_1 + D_2^T D_2 \otimes I_{M_1})
$$

where $D_d$ are difference operators, and $\tau > 0$ controls smoothness.

**Prior specification**:

$$
\boldsymbol{\beta} \mid \tau \sim \mathcal{N}(\mathbf{0}, Q^{-1}), \quad \tau \sim \text{Gamma}(2, 0.5)
$$

This induces a smooth GP-like prior on $f$:

$$
\mathbf{f} \mid \tau \sim \mathcal{N}(\mathbf{0}, \Phi Q^{-1} \Phi^T)
$$

#### **Prior Configuration Parameters**

Each `Prior2D` subclass takes two critical parameters:

1. `prior_type`: Specifies the data structure:
   - `'global'`: Single matrix (e.g., $f(a,b)$ for the whole population)
   - `'partial'`: Stratified by participant subgroup only → matrices $\delta^k_{a,b}$ (assumes symmetric contacts)
   - `'full'`: Stratified by both participant *and* contact subgroup → matrices $\delta^{k,\ell}_{a,b}$ (full heterogeneity)

2. `transform`: Real-to-simplex transformation for offsets (ignored for `'global'` type):
   - `'alr'` (Additive Log-Ratio): Uses first subgroup as reference
   - `'clr'` (Centered Log-Ratio / softmax): Symmetric treatment of all subgroups
   - `'ilr'` (Isometric Log-Ratio): Orthonormal transformation—**recommended** for computational stability

**Why transforms?** The offsets must satisfy compositional constraints (deviations from the global rate must "balance out" across subgroups). Transformations map unconstrained latent variables to this constrained space while preserving prior smoothness.


In [ ]:
priors = {
    # Prior for the global rate
    "rate": PSpline2D(
        prior_type="global",  # Symmetric contact matrix
        order=2,  # For PSpline priors we can also specify the order of differences in the penalty matrix
    ),
    # Prior for the contact rate offsets
    "subgroup": PSpline2D(prior_type="partial", order=2, transform="ilr"),
}

print("Prior configured:")
print(f"  Rate: {type(priors["rate"]).__name__}")
print(f"  Subgroup: {type(priors["subgroup"]).__name__}")

**Interpretation**:
- The `"rate"` prior enforces smoothness on the population-averaged contact structure
- The `"subgroup"` prior allows each subgroup's contact intensity to deviate smoothly from this average
- `order=2` penalizes curvature (second-order differences), producing very smooth surfaces; use `order=1` for less aggressive smoothing


We are now ready to initialize the `HiBRCfine` model, which takes three components:
1. `dataloader`: The `DataLoader` object containing the processed data
2. `priors`: A dictionary specifying priors for the global contact rate and subgroup offsets
3. `likelihood`: The likelihood function (options: `'negbin'` or `'poisson'`)


In [ ]:
model = HiBRCfine(
    dataloader=dataloader,
    priors=priors,
    likelihood="negbin",  # Contact counts follow Poisson distribution
)

print("BRCfine model initialized:")
print(f"  Likelihood: {model.likelihood}")
print(f"  Prior: {type(priors["rate"]).__name__}")
print(f"  Partial prior for subgroup: {type(priors["subgroup"]).__name__}")

# Print the parameter dimensions of the model
model.print_model_shape()

### 3.3 Inference with Stochastic Variational Inference (SVI)

With the model specified, we now perform inference to estimate the posterior distribution of all parameters. As shown by `print_model_shape()` above, the `HiBRCfine` model is high-dimensional—with hundreds to thousands of parameters depending on the number of basis functions and subgroups.

While full Bayesian inference with Markov Chain Monte Carlo (MCMC) is possible, it becomes computationally expensive for such high-dimensional models. We use Stochastic Variational Inference (SVI) instead, which offers speed and scalability at the cost of producing an *approximate* posterior that may underestimate uncertainty.


#### **Configuring the Variational Family (Guide)**

For SVI, we specify a **guide**—the parametric family used to approximate the posterior. We use NumPyro's `AutoNormal` guide, which constructs a **mean-field approximation**:

$$
q(\boldsymbol{\theta}) = \prod_{i=1}^{D} \mathcal{N}(\theta_i \mid \mu_i, \sigma_i^2)
$$

Each parameter $\theta_i$ is approximated by an independent Gaussian with learnable mean $\mu_i$ and standard deviation $\sigma_i$. This factorization assumption (independence between parameters) enables fast optimization but may underestimate uncertainty.

**Initialization strategy**: Variational inference is sensitive to initial parameter values. We use `init_to_mean`, which initializes each $\mu_i$ to the mean of its prior distribution.

**Alternative guides** (not used here):
- `AutoDelta`: Point estimates only (no uncertainty quantification)
- `AutoLowRankMultivariateNormal`: Captures some correlations via low-rank covariance (slower but more accurate)
- `AutoMultivariateNormal`: Full covariance (computationally prohibitive for large models)

In [ ]:
from numpyro.infer.autoguide import AutoNormal
from numpyro.infer.initialization import init_to_mean

# Define the variational family (guide) and initialization strategy
guide = AutoNormal(model.model, init_loc_fn=init_to_mean)

#### **Running SVI Optimization**

We execute SVI using the `run_inference_svi()` method, which wraps NumPyro's SVI implementation with sensible defaults for contact matrix estimation.

**Key parameters**:

- **`prng_key`**: JAX random key for reproducible stochastic optimization (always required in JAX workflows)
- **`guide`**: The variational approximation (our `AutoNormal` guide from above)
- **`num_steps`**: Total optimization iterations
  - **Tutorial/debugging**: 5,000–10,000 steps
  - **Production**: 20,000–50,000 steps
  - Monitor the ELBO (Evidence Lower Bound) loss—it should plateau near convergence
- **`peak_lr`**: Peak learning rate for the onecycle schedule (default: 0.01)
  - The optimizer uses a triangular learning rate schedule: warmup → peak → decay
  - Reduce to 0.001 if ELBO shows large oscillations

**Monitoring convergence**: The ELBO should increase monotonically (with small fluctuations) and plateau. If it diverges or oscillates wildly, reduce `peak_lr` or increase `num_steps`.


In [ ]:
print("Running SVI inference...")
print("This may take a few minutes...")

# Pseudo-random number generator key (JAX)
prng_key = random.PRNGKey(42)

model.run_inference_svi(
    prng_key=prng_key,
    guide=guide,
    num_steps=10_000,  # Number of optimization steps
    peak_lr=0.01,  # Learing rate
)

print("Inference complete!")

#### Side Note: Full Bayesian Inference with MCMC

For more accurate uncertainty quantification, we can use Markov Chain Monte Carlo (MCMC) instead of SVI:
```python
# Alternative: Use MCMC for exact posterior (slower but more accurate)
model.run_inference_mcmc(
    rng_key=random.PRNGKey(42),
    num_warmup=1000,      # Burn-in period
    num_samples=2000,     # Posterior samples
    num_chains=4,         # Parallel chains
)
```


## IV. Analyzing the Posterior

After inference, we extract posterior summaries to interpret the estimated contact patterns. Depending on the inference method used, posterior samples are stored in:

- **SVI**: `model._svi_result` (variational parameters + guide samples)
- **MCMC**: `model._mcmc_result` (NUTS samples from all chains)

The `analysis` module provides two key classes for post-inference analysis:

1. `ModelSummariserBRC`: Computes posterior summaries (medians, credible intervals) for contact matrices
2. `ModelEvaluatorBRC`: Evaluates model accuracy against ground truth (for synthetic data validation)

### 4.1 Extracting Posterior Summaries

The `ModelSummariserBRC` class provides a unified interface for summarizing posterior distributions, regardless of whether you used SVI or MCMC for inference.

**Key methods**:

- **`summarise_cint(alpha=0.05)`**: Contact intensity matrices $m^k_{a,b}$ with $(1-\alpha)$ credible intervals
  - Returns median + lower/upper bounds for each subgroup
  - Incorporates population structure via $\log P_b$ offset
  
- **`summarise_mcint(alpha=0.05)`**: Marginal contact intensities (sum over contacted ages)
  - Computes $\sum_b m^k_{a,b}$ for each age $a$
  - Useful for age-specific contact burden analysis


In [ ]:
# Initialize summariser (may take a few seconds)
summariser = ModelSummariserBRC(model)

In [ ]:
# Global contact intensity summaries
summary_cint = summariser.summarise_cint(alpha=0.05)

print(" Type of summary_cint: ", type(summary_cint).__name__)
print(" Keys: ", summary_cint.keys())
print(" Subgroups: ", summary_cint["subgroup"].keys())
print(" Values shape: ", summary_cint["subgroup"]["high"].shape)  # (3, 81, 81)

**Understanding the output structure**: The `summarise_cint()` method returns a nested dictionary organized by stratification level. For `HiBRCfine` with `prior_type='partial'`, the structure is:

```python
{
    'subgroup': {
        'low':  np.ndarray of shape (3, 81, 81),  # [lower, median, upper]
        'mid':  np.ndarray of shape (3, 81, 81),
        'high': np.ndarray of shape (3, 81, 81)
    }
}
```

**Array dimensions**:
- Axis 0 (size 3): Summary statistics for the credible interval
  - Index 0: Lower bound (2.5th percentile for `alpha=0.05`)
  - Index 1: Median (50th percentile)
  - Index 2: Upper bound (97.5th percentile)
- Axes 1–2 (size 81×81): Contact intensity matrix $m^k_{a,b}$ for ages 0–80


We can compute summaries for the marginal contact intensities as well:


In [ ]:
summary_mcint = summariser.summarise_mcint(alpha=0.05)

print(" Type of summary_mcint: ", type(summary_mcint).__name__)
print(" Keys: ", summary_mcint.keys())
print(" Subgroups: ", summary_mcint["subgroup"].keys())
print(" Values shape: ", summary_mcint["subgroup"]["high"].shape)  # (3, 81, 81)

### 4.2 Visualize Estimated Contact Matrices

We now visualize the posterior median contact intensity matrices.


In [ ]:
# Plot estimated contact intensity (median)
charts = []
# Sort by the desired order: low, mid, high
for label in LABELS:
    matrix = summary_cint["subgroup"][label]
    chart = plot_mosaic(
        matrix[1],
        title=f"Estimated Contact Intensity: {label.capitalize()}",
        zlabel="Intensity",
    ).properties(width=200, height=200)
    charts.append(chart)

chart_est = alt.hconcat(*charts).resolve_scale(color="independent")
chart_est

We can visually compare the true, empirical, and estimated contact intensity matrices.


In [ ]:
chart_true & chart_emp & chart_est

We can also inspect marginal contact intensities for each subpopulation.


In [ ]:
from typing import Dict


def mcint_summary_to_df(summary: Dict) -> pd.DataFrame:
    """
    Helper to process posterior marginal contact intensity summaries

    Parameters
    ----------
    summary: Dict
        The output of summarise_mcint() method in ModelSummariserBRC

    Returns
    -------
    pd.DataFrame
        The posterior summary in a DataFrame format
    """
    dfs = []
    for label in LABELS:
        summary_cat = summary["subgroup"][label]
        df_mcint = pd.DataFrame(
            {
                "age": np.arange(summary_cat[1].shape[0]),
                "subgroup": label,
                "mid": summary_cat[1],
                "lower": summary_cat[0],
                "upper": summary_cat[2],
            }
        )
        dfs.append(df_mcint)

    return pd.concat(dfs, ignore_index=True)


df_summary_mcint = mcint_summary_to_df(summary_mcint)

In [ ]:
x_axis = alt.X(
    "age:Q", title="Age of contacting individual", scale=alt.Scale(domain=[0, 84])
)

lines = (
    alt.Chart(df_summary_mcint)
    .mark_line()
    .encode(
        x=x_axis, y=alt.Y("mid:Q", title="Marginal contact intensity"), color="subgroup"
    )
    .properties(title="Estimated Mean Contacts per Person by Age and Subgroup")
)

bands = (
    alt.Chart(df_summary_mcint)
    .mark_area(opacity=0.3)
    .encode(x=x_axis, y="lower", y2="upper", color="subgroup")
)

chart = alt.layer(lines, bands).properties(
    title="Estimated Mean Contacts per Person by Age and Subgroup"
)

chart

## V. Evaluate the Accuracy of the Inference

Since we are working with synthetic data, we can evaluate the accuracy of our inference by comparing the estimated contact intensity matrices with the true matrices used to generate the data. The `ModelEvaluatorBRC` class computes error metrics such as root mean squared error (RMSE) and mean absolute error (MAE). It also evaluates uncertainty quantification through interval scores and posterior coverage probabilities.

### 5.1 Initialize the Evaluator

We create an evaluator instance by providing the summariser and the ground truth contact matrices:

**Important**: The structure of `cint_matrix_true` must match your model type:
- For `HiBRCfine` with `prior_type='partial'`: Use `{"subgroup": contact_matrices}` where `contact_matrices` is a dict of matrices by subgroup
- For standard `BRCfine`: Pass a single `np.ndarray` directly

In [ ]:
# Step 11: Initialize evaluator with true contact matrix
evaluator = ModelEvaluatorBRC(
    summariser=summariser,
    cint_matrix_true={"subgroup": contact_matrices},  # Ground truth for comparison
)

### 5.2 Compute Evaluation Metrics

We now compute evaluation metrics:

**Understanding the output**: The `evaluate()` method returns a DataFrame with two types of evaluations:

1. Contact intensity (`cint`): Full age-age matrix $m^k_{a,b}$
2. Marginal contact intensity (`mcint`): $\sum_b m^k_{a,b}$


In [ ]:
metrics = evaluator.evaluate()
metrics

#### Accuracy Metrics

- **`rmse`** (Root Mean Squared Error): 
  $$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^n (y_i - \hat{y}_i)^2}$$
  Measures average prediction error. Lower is better. Scale: same as contact intensity units.

- **`mae`** (Mean Absolute Error):
  $$\text{MAE} = \frac{1}{n}\sum_{i=1}^n |y_i - \hat{y}_i|$$
  Average absolute deviation. More robust to outliers than RMSE. Lower is better.

- **`mape`** (Mean Absolute Percentage Error):
  $$\text{MAPE} = \frac{100}{n}\sum_{i=1}^n \left|\frac{y_i - \hat{y}_i}{y_i}\right|$$
  Relative error as percentage. Scale-invariant metric. Lower is better.

#### Uncertainty Quantification Metrics

- **`interval_score`**: Negatively oriented proper scoring rule that evaluates both interval width and coverage:
  $$\text{IS}_\alpha(y, [l, u]) = (u - l) + \frac{2}{\alpha}(l - y)\mathbb{1}_{y < l} + \frac{2}{\alpha}(y - u)\mathbb{1}_{y > u}$$
  - Penalizes wide intervals (first term)
  - Penalizes coverage failures (second and third terms)
  - Lower is better (well-calibrated narrow intervals score best)

- **`coverage`**: Percentage of true values falling within the $(1-\alpha)$ credible intervals:
  $$\text{Coverage} = \frac{100}{n}\sum_{i=1}^n \mathbb{1}_{l_i \leq y_i \leq u_i}$$
  - Should be close to $(1 - \alpha) \times 100$ (e.g., 95% for `alpha=0.05`)
  - Lower coverage indicates underestimated uncertainty
  - Higher coverage may indicate overly conservative intervals
